# Metropolitan Driverless Transit — Evolutionary Simulation Lab
Executable research prototype with local HTML front end. Python 3.10+; simulation and server require only the standard library. Jupyter is required to open this notebook. Run cells in order.

**Scope:** synthetic single line, two directions. This is not certified railway control software. Energy is a traction proxy, not a validated engineering model. No trained LLM or automatic API calls are included. LLM assistance is an explicit, optional JSON-parameter import.

The notebook contains the complete backend, server and front end, and writes them into `metro_runtime` so it also works independently of the ZIP.

## 1. Materialise the complete application
This cell writes readable source files beside the notebook. Existing runtime files with the same names are replaced.

In [1]:
from pathlib import Path
import sys, json
RUNTIME = Path.cwd() / "metro_runtime"
RUNTIME.mkdir(exist_ok=True)
FILES = {'transit.py': '"""Deterministic, discrete-time metro research simulator. Units: minutes, km, kWh."""\nfrom dataclasses import dataclass, asdict\nimport random, math, statistics, json, time\n\n@dataclass(frozen=True)\nclass Config:\n    stations: int = 10\n    duration: int = 120\n    fleet: int = 16\n    capacity: int = 350\n    demand: float = 10.0\n    surge: float = 2.0\n    disruption_start: int = 40\n    disruption_end: int = 65\n    slow_factor: float = 1.8\n    seed: int = 42\n    def validate(self):\n        for k,lo,hi in [(\'stations\',4,20),(\'duration\',30,240),(\'fleet\',4,30),(\'capacity\',50,1000),(\'seed\',0,1000000)]:\n            v=getattr(self,k)\n            if type(v) is not int or not lo<=v<=hi: raise ValueError(f\'{k} must be an integer in [{lo}, {hi}]\')\n        for k,lo,hi in [(\'demand\',0,50),(\'surge\',1,5),(\'slow_factor\',1,3)]:\n            v=getattr(self,k)\n            if isinstance(v,bool) or not isinstance(v,(float,int)) or not math.isfinite(v) or not lo<=v<=hi: raise ValueError(f\'Invalid {k}\')\n        if not (type(self.disruption_start) is int and type(self.disruption_end) is int and 0<=self.disruption_start<=self.disruption_end<=self.duration): raise ValueError(\'Invalid disruption interval\')\n        return self\n\n@dataclass(frozen=True)\nclass Policy:\n    headway: float = 3.0\n    crowd_weight: float = 0.0\n    hold: float = 0.0\n    eco: float = 0.0\n    def validate(self):\n        for key,lo,hi in [(\'headway\',2,6),(\'crowd_weight\',0,1),(\'hold\',0,2),(\'eco\',0,1)]:\n            v=getattr(self,key)\n            if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not lo<=v<=hi: raise ValueError(f\'Invalid policy {key}\')\n        return self\n\ndef arrivals(c):\n    """Exogenous demand sampled once, independent of scheduling decisions."""\n    rng=random.Random(c.seed); out=[]\n    for t in range(c.duration):\n        batch=[]\n        for d in range(2):\n            for s in range(c.stations):\n                if (d==0 and s==c.stations-1) or (d==1 and s==0): continue\n                rate=c.demand*(1.0+0.4*math.sin(math.pi*t/c.duration))\n                if s==c.stations//2 and c.disruption_start<=t<c.disruption_end: rate*=c.surge\n                # Poisson draws, Knuth algorithm. Maximum permitted rate is modest.\n                n=0; p=1.; limit=math.exp(-rate)\n                while p>limit: p*=rng.random(); n+=1\n                for _ in range(max(0,n-1)):\n                    dest=rng.randrange(s+1,c.stations) if d==0 else rng.randrange(0,s)\n                    batch.append((d,s,dest,t))\n        out.append(batch)\n    return out\n\ndef simulate(c=Config(), p=Policy(), trace=True):\n    c.validate(); p.validate(); demand=arrivals(c)\n    queues={(d,s):[] for d in range(2) for s in range(c.stations)}\n    trains=[dict(id=i,station=0 if i%2==0 else c.stations-1,d=i%2,ready=0.,on=[],moving=False,origin=None,depart=None,arrival=None) for i in range(c.fleet)]\n    # Directed block occupancy and platform departure separation are independently enforced.\n    block_free={}; last_depart={}; last_terminal={0:-100.,1:-100.}\n    waits=[]; boarded=completed=generated=0; passenger_minutes=0; energy=0.; pkm=0.; blocked=0; history=[]; departures=[]\n    step=.25\n    for tick in range(c.duration*4):\n        t=tick*step\n        if tick%4==0:\n            for d,s,dest,a in demand[tick//4]: queues[d,s].append((dest,a)); generated+=1\n        for tr in trains:\n            if tr[\'moving\'] and t+1e-9>=tr[\'arrival\']:\n                tr[\'station\']+=1 if tr[\'d\']==0 else -1\n                tr[\'moving\']=False\n                remaining=[]; alight=0\n                for dest,a in tr[\'on\']:\n                    if dest==tr[\'station\']: completed+=1; alight+=1\n                    else: remaining.append((dest,a))\n                tr[\'on\']=remaining\n                terminal=tr[\'station\'] in (0,c.stations-1)\n                if terminal:\n                    tr[\'d\']=0 if tr[\'station\']==0 else 1\n                    tr[\'ready\']=t+2.0\n                else: tr[\'ready\']=t+.5+p.hold*min(1,alight/c.capacity)\n        # Stable order avoids platform arbitration depending on dictionary iteration.\n        for tr in trains:\n            if tr[\'moving\'] or t<tr[\'ready\']: continue\n            s=tr[\'station\']; d=tr[\'d\']; nxt=s+(1 if d==0 else -1)\n            if not 0<=nxt<c.stations: continue\n            terminal=s in (0,c.stations-1)\n            pressure=len(queues[d,s])/c.capacity\n            target=max(2.,p.headway-p.crowd_weight*min(2.,pressure))\n            if terminal and t-last_terminal[d]<target: continue\n            key=(d,s)\n            if t<block_free.get(key,0.) or t-last_depart.get(key,-100.)<1.5:\n                blocked+=1; continue\n            q=queues[d,s]; take=min(c.capacity-len(tr[\'on\']),len(q)); selected=q[:take]; del q[:take]\n            for dest,a in selected: waits.append(t-a)\n            boarded+=take; tr[\'on\'].extend(selected)\n            slow=c.disruption_start<=t<c.disruption_end and s==c.stations//2\n            runtime=2.*(c.slow_factor if slow else 1.)*(1+.25*p.eco)\n            # Illustrative traction proxy; no regenerative-braking credit.\n            load=len(tr[\'on\']); energy+=(6.+load*.006)*(1-.18*p.eco)\n            pkm+=load*1.2\n            tr.update(moving=True,origin=s,depart=t,arrival=t+runtime)\n            block_free[key]=tr[\'arrival\']+.5; last_depart[key]=t\n            if terminal: last_terminal[d]=t\n            if trace: departures.append(dict(time=t,train=tr[\'id\'],direction=d,station=s,load=load))\n        total_queue=sum(map(len,queues.values())); passenger_minutes+=total_queue*step\n        if trace and tick%4==0:\n            positions=[]\n            for tr in trains:\n                pos=tr[\'station\']\n                if tr[\'moving\']: pos+=(1 if tr[\'d\']==0 else -1)*min(1,(t-tr[\'depart\'])/(tr[\'arrival\']-tr[\'depart\']))\n                positions.append(dict(id=tr[\'id\'],d=tr[\'d\'],position=pos,load=len(tr[\'on\'])))\n            history.append(dict(time=t,queues=[len(queues[d,s]) for d in range(2) for s in range(c.stations)],trains=positions))\n    sorted_waits=sorted(waits)\n    mean=statistics.mean(waits) if waits else None\n    p95=sorted_waits[min(len(waits)-1,int(.95*len(waits)))] if waits else None\n    stranded=sum(map(len,queues.values())); onboard=sum(len(tr[\'on\']) for tr in trains)\n    assert generated==completed+stranded+onboard, \'Passenger conservation failed\'\n    # Includes observed waiting of censored passengers, unlike boarded-only mean.\n    objective=passenger_minutes/max(1,generated)+energy/max(1,generated)*.2\n    return dict(config=asdict(c),policy=asdict(p),metrics=dict(generated=generated,boarded=boarded,completed=completed,waiting_at_end=stranded,onboard_at_end=onboard,mean_boarded_wait=mean,p95_boarded_wait=p95,queue_passenger_minutes=passenger_minutes,observed_wait_per_arrival=passenger_minutes/max(1,generated),energy_kwh_proxy=energy,passenger_km=pkm,energy_per_passenger_km=energy/max(1,pkm),blocked_departure_attempts=blocked,objective=objective),history=history,departures=departures)\n\ndef optimise(c, population=10, generations=6, seeds=None, initial=None):\n    """Budgeted elitist GA + coordinate local search; no test-seed access."""\n    if not 4<=population<=24 or not 1<=generations<=15: raise ValueError(\'Search budget out of range\')\n    seeds=seeds or [c.seed,c.seed+1]; rng=random.Random(c.seed+999); cache={}; curve=[]\n    bounds=[(2,6),(0,1),(0,2),(0,1)]\n    def repair(v): return [round(min(hi,max(lo,x)),4) for x,(lo,hi) in zip(v,bounds)]\n    def evaluate(v):\n        key=tuple(v)\n        if key not in cache:\n            cache[key]=statistics.mean(simulate(Config(**{**asdict(c),\'seed\':s}),Policy(*v),False)[\'metrics\'][\'objective\'] for s in seeds)\n        return cache[key]\n    pop=[[3,0,0,0]]\n    if initial: initial.validate(); pop.append(list(asdict(initial).values()))\n    while len(pop)<population: pop.append([rng.uniform(lo,hi) for lo,hi in bounds])\n    start=time.perf_counter()\n    for g in range(generations):\n        pop=sorted(pop,key=evaluate); best=pop[0]; curve.append(dict(generation=g,best=evaluate(best)))\n        children=pop[:2]\n        # Local coordinate refinement around elite.\n        v=best.copy(); j=g%4; v[j]+=rng.choice([-1,1])*(bounds[j][1]-bounds[j][0])*.08; children.append(repair(v))\n        while len(children)<population:\n            a,b=rng.sample(pop[:max(3,population//2)],2)\n            child=[a[j] if rng.random()<.5 else b[j] for j in range(4)]\n            j=rng.randrange(4); child[j]+=rng.gauss(0,(bounds[j][1]-bounds[j][0])*.15)\n            children.append(repair(child))\n        pop=children\n    best=min(pop,key=evaluate)\n    return dict(policy=asdict(Policy(*best)),training_objective=evaluate(best),curve=curve,evaluations=len(cache)*len(seeds),seconds=time.perf_counter()-start,training_seeds=seeds)\n\ndef experiment(c, seeds=(101,102,103), population=8, generations=4, initial=None):\n    if set(seeds)&{c.seed,c.seed+1}: raise ValueError(\'Test seeds must be independent\')\n    trained=optimise(c,population,generations,initial=initial)\n    policies={\'Fixed timetable\':Policy(),\'Demand reactive\':Policy(3,.8,0,0),\'Evolution + local search\':Policy(**trained[\'policy\'])}\n    if initial: policies[\'Imported LLM rule\']=initial\n    rows=[]\n    for seed in seeds:\n        for name,p in policies.items(): rows.append(dict(method=name,seed=seed,**simulate(Config(**{**asdict(c),\'seed\':seed}),p,False)[\'metrics\']))\n    return dict(training=trained,rows=rows,notes=\'Synthetic single line, two directions; training/test seeds disjoint. Energy is a proxy. No operational certification.\')\n', 'server.py': '"""Loopback-only dashboard server; bounded requests and serial compute."""\nfrom http.server import HTTPServer, BaseHTTPRequestHandler\nfrom pathlib import Path\nimport json, logging\nfrom transit import Config, Policy, simulate, optimise, experiment\nROOT=Path(__file__).parent\nclass Handler(BaseHTTPRequestHandler):\n    def reply(self,status,data,kind=\'application/json\'):\n        raw=data.encode() if isinstance(data,str) else json.dumps(data,allow_nan=False).encode()\n        self.send_response(status); self.send_header(\'Content-Type\',kind); self.send_header(\'Content-Length\',str(len(raw))); self.send_header(\'X-Content-Type-Options\',\'nosniff\'); self.end_headers(); self.wfile.write(raw)\n    def do_GET(self):\n        if self.path in (\'/\',\'/index.html\'): self.reply(200,(ROOT/\'index.html\').read_text(),\'text/html; charset=utf-8\')\n        elif self.path==\'/health\': self.reply(200,{\'status\':\'ok\'})\n        else: self.reply(404,{\'error\':\'Not found\'})\n    def do_POST(self):\n        if self.headers.get(\'Host\') not in (\'127.0.0.1:8765\',\'localhost:8765\'): return self.reply(403,{\'error\':\'Invalid host\'})\n        origin=self.headers.get(\'Origin\')\n        if origin and origin not in (\'http://127.0.0.1:8765\',\'http://localhost:8765\'): return self.reply(403,{\'error\':\'Invalid origin\'})\n        try:\n            n=int(self.headers.get(\'Content-Length\',\'0\'))\n            if not 0<n<=12000: raise ValueError(\'Body must be 1–12000 bytes\')\n            body=json.loads(self.rfile.read(n)); c=Config(**body.get(\'config\',{})).validate()\n            p=Policy(**body.get(\'policy\',{})).validate()\n            if self.path==\'/api/simulate\': result=simulate(c,p)\n            elif self.path==\'/api/optimise\':\n                result=optimise(c,body.get(\'population\',8),body.get(\'generations\',4),initial=p)\n                result[\'simulation\']=simulate(c,Policy(**result[\'policy\']))\n            elif self.path==\'/api/experiment\': result=experiment(c,population=6,generations=3,initial=p)\n            else: return self.reply(404,{\'error\':\'Unknown endpoint\'})\n            self.reply(200,result)\n        except (ValueError,TypeError,KeyError) as exc: self.reply(400,{\'error\':str(exc)})\n        except Exception:\n            logging.exception(\'Request failed\'); self.reply(500,{\'error\':\'Internal simulation error; see server log\'})\n\ndef serve():\n    logging.basicConfig(level=logging.INFO)\n    print(\'Open http://127.0.0.1:8765 — Ctrl+C to stop\',flush=True)\n    HTTPServer((\'127.0.0.1\',8765),Handler).serve_forever()\nif __name__==\'__main__\': serve()\n', 'index.html': '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Metro Evolution Lab</title>\n<style>body{margin:0;background:#091322;color:#e1eafa;font:15px system-ui}main{max-width:1250px;margin:auto;padding:28px}h1{font-size:32px;margin-bottom:6px}p{color:#a9bbd0}section{background:#132338;border:1px solid #29415a;border-radius:14px;padding:20px;margin:18px 0}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px}label{display:block;color:#a9bbd0;font-size:13px}input,textarea,button,select{box-sizing:border-box;background:#091728;color:#eef;border:1px solid #38516b;border-radius:7px;padding:9px;width:100%;margin-top:6px}button{background:#137b83;cursor:pointer;font-weight:600}button:disabled{opacity:.4}canvas{width:100%;height:320px;background:#0c192a;border-radius:10px}small{color:#9cb1c8}.metric{font-size:23px;font-weight:650}.bar{display:flex;gap:12px;margin-top:14px}table{border-collapse:collapse;width:100%;font-size:13px}th,td{text-align:left;padding:9px;border-bottom:1px solid #30435c}#status{white-space:pre-wrap;color:#68e0ca}textarea{min-height:85px}a{color:#60ddd0}</style>\n<main><small>RESEARCH SIMULATION • SYNTHETIC METROPOLITAN LINE</small><h1>Metro Evolution Lab</h1><p>Explore demand surges, speed restrictions and evolutionary dispatch policies. One line, two directions, dedicated tracks. Energy values are illustrative estimates.</p>\n<section><div class="grid" id="controls"></div><div class="bar"><button id="run">Run simulation</button><button id="opt">Evolve dispatch policy</button><button id="exp">Held-out comparison</button><button id="download">Export JSON</button></div><p id="status" role="status">Ready. Start with the default scenario.</p></section>\n<section><h2>Policy parameters / LLM rule import</h2><p>An external LLM may propose this bounded JSON policy. Importing parameters never executes generated Python.</p><textarea id="policy">{"headway":3,"crowd_weight":0,"hold":0,"eco":0}</textarea><small>headway: 2–6 minutes; crowd_weight: 0–1; hold: 0–2 minutes; eco: 0–1.</small></section>\n<section><div class="grid" id="metrics"></div></section><section><h2>Network playback</h2><canvas id="network" width="1160" height="320"></canvas><input id="frame" type="range" min="0" max="119" value="0"><div class="bar"><button id="play">Play / pause</button><span id="clock"></span></div><small>Upper track: outbound; lower track: inbound. Circles: trains. Station bars: queued passengers.</small></section>\n<section><h2>Passenger queues over time</h2><canvas id="chart" width="1160" height="320"></canvas></section><section><h2>Independent evaluation</h2><div id="results">Run a held-out comparison to inspect per-seed results.</div></section>\n<section><h2>Model boundary</h2><p>Fixed 15-second steps; identical exogenous demand for each policy at a given seed. Tracks use conservative exclusive directed blocks plus 90-second departure separation. Terminal turnaround is two minutes. No signalling, braking curves, junctions, station platform capacity, train failures, transfers or certified safety assessment are modelled. Optimisation is scalar, not a Pareto solver. Hold is an alighting-dependent dwell extension. Completed journeys, boarded-only wait and unfinished passengers are reported separately.</p></section></main>\n<script>\nconst $=id=>document.getElementById(id);const defaults={stations:10,duration:120,fleet:16,capacity:350,demand:10,surge:2,disruption_start:40,disruption_end:65,slow_factor:1.8,seed:42};\nfor(const [k,v] of Object.entries(defaults)){let l=document.createElement(\'label\');l.textContent=k.replaceAll(\'_\',\' \');let i=document.createElement(\'input\');i.id=k;i.type=\'number\';i.value=v;i.step=[\'demand\',\'surge\',\'slow_factor\'].includes(k)?\'.1\':\'1\';l.append(i);$(\'controls\').append(l)}\nlet sim=null,last=null,timer=null;\nfunction plot(canvas,points,color){const x=canvas.getContext(\'2d\');x.clearRect(0,0,canvas.width,canvas.height);if(!points.length)return;let max=Math.max(1,...points);x.strokeStyle=color;x.lineWidth=3;x.beginPath();points.forEach((p,i)=>{let a=45+i/(Math.max(1,points.length-1))*(canvas.width-65),b=280-p/max*240;i?x.lineTo(a,b):x.moveTo(a,b)});x.stroke();x.fillStyle=\'#a9bbd0\';x.fillText(\'Queued passengers — maximum \'+max,45,22);x.fillText(\'Simulation minutes →\',45,307)}\nfunction draw(){if(!sim)return;const h=sim.history[+$(\'frame\').value];if(!h)return;const c=$(\'network\'),x=c.getContext(\'2d\'),n=sim.config.stations; x.clearRect(0,0,c.width,c.height);for(let d=0;d<2;d++){let y=110+d*130;x.strokeStyle=d?\'#b89bfa\':\'#4cd4cc\';x.lineWidth=4;x.beginPath();x.moveTo(50,y);x.lineTo(1110,y);x.stroke();for(let s=0;s<n;s++){let xx=50+s*1060/(n-1);x.fillStyle=\'#b8cadd\';x.fillRect(xx-5,y-7,10,14);x.fillText(\'S\'+s,xx-8,y+27);let q=h.queues[d*n+s];x.fillStyle=\'#edb869\';x.fillRect(xx-8,y-15-Math.min(55,q/8),16,Math.min(55,q/8));x.fillText(q,xx-9,y-75)}}for(const tr of h.trains){let xx=50+tr.position*1060/(n-1),y=110+tr.d*130;x.fillStyle=tr.d?\'#b89bfa\':\'#4cd4cc\';x.beginPath();x.arc(xx,y,10,0,7);x.fill();x.fillStyle=\'#fff\';x.fillText(tr.id,xx-5,y+4)}$(\'clock\').textContent=\'Minute \'+h.time}\nfunction render(s){sim=s;$(\'frame\').max=s.history.length-1;$(\'frame\').value=0;draw();plot($(\'chart\'),s.history.map(h=>h.queues.reduce((a,b)=>a+b,0)),\'#54d9cc\');$(\'metrics\').replaceChildren();for(const key of [\'mean_boarded_wait\',\'p95_boarded_wait\',\'waiting_at_end\',\'completed\',\'energy_kwh_proxy\',\'observed_wait_per_arrival\']){const e=document.createElement(\'div\');const label=document.createElement(\'small\');label.textContent=key.replaceAll(\'_\',\' \');const value=document.createElement(\'div\');value.className=\'metric\';value.textContent=s.metrics[key]===null?\'N/A\':Number(s.metrics[key]).toFixed(2);e.append(label,value);$(\'metrics\').append(e)}}\nasync function request(action){for(const id of [\'run\',\'opt\',\'exp\'])$(id).disabled=true;$(\'status\').textContent=\'Computing locally…\';try{let config={};for(let k of Object.keys(defaults))config[k]=Number($(k).value);let policy=JSON.parse($(\'policy\').value);const r=await fetch(\'/api/\'+action,{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({config,policy})});const data=await r.json();if(!r.ok)throw Error(data.error);last=data;if(action===\'simulate\')render(data);else if(action===\'optimise\'){render(data.simulation);$(\'policy\').value=JSON.stringify(data.policy);$(\'status\').textContent=\'Search complete: \'+data.evaluations+\' training simulations in \'+data.seconds.toFixed(1)+\' s\';return}else{let t=document.createElement(\'table\'),head=t.insertRow();for(let s of [\'Method\',\'Seed\',\'Observed wait / arrival\',\'Completed\',\'Waiting at end\',\'Energy proxy\']){let th=document.createElement(\'th\');th.textContent=s;head.append(th)}for(let row of data.rows){let tr=t.insertRow();for(let v of [row.method,row.seed,row.observed_wait_per_arrival.toFixed(2),row.completed,row.waiting_at_end,row.energy_kwh_proxy.toFixed(1)])tr.insertCell().textContent=v}$(\'results\').replaceChildren(t)}$(\'status\').textContent=\'Complete. Results are synthetic; export JSON for reproducibility.\'}catch(e){$(\'status\').textContent=\'Error: \'+e.message}finally{for(let id of [\'run\',\'opt\',\'exp\'])$(id).disabled=false}}\n$(\'run\').onclick=()=>request(\'simulate\');$(\'opt\').onclick=()=>request(\'optimise\');$(\'exp\').onclick=()=>request(\'experiment\');$(\'frame\').oninput=draw;$(\'play\').onclick=()=>{if(timer){clearInterval(timer);timer=null}else timer=setInterval(()=>{if(!sim)return;$(\'frame\').value=(+$(\'frame\').value+1)%sim.history.length;draw()},100)};$(\'download\').onclick=()=>{if(!last)return;let u=URL.createObjectURL(new Blob([JSON.stringify(last,null,2)],{type:\'application/json\'}));let a=document.createElement(\'a\');a.href=u;a.download=\'metro_experiment.json\';a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)};\n</script></html>\n', 'test_transit.py': "import unittest\nfrom transit import *\nclass TransitTests(unittest.TestCase):\n def test_reproducible(self):\n  c=Config(duration=30,disruption_start=10,disruption_end=20)\n  self.assertEqual(simulate(c),simulate(c))\n def test_zero_demand(self):\n  r=simulate(Config(duration=30,demand=0,disruption_start=10,disruption_end=20))\n  self.assertEqual(r['metrics']['generated'],0)\n  self.assertIsNone(r['metrics']['mean_boarded_wait'])\n def test_separation_and_capacity(self):\n  c=Config(duration=30,fleet=30,disruption_start=10,disruption_end=20)\n  r=simulate(c,Policy(2,1,0,0)); last={}\n  for e in r['departures']:\n   k=e['direction'],e['station']\n   if k in last:\n    previous=last[k]; slow=c.disruption_start<=previous<c.disruption_end and e['station']==c.stations//2\n    self.assertGreaterEqual(e['time']-previous,2*(c.slow_factor if slow else 1)+.5-1e-8)\n   last[k]=e['time'];self.assertLessEqual(e['load'],c.capacity)\n def test_validation(self):\n  for kwargs in [dict(demand=float('nan')),dict(stations=2),dict(fleet=True)]:\n   with self.assertRaises(ValueError): Config(**kwargs).validate()\n  with self.assertRaises(ValueError):Policy(headway=0).validate()\n def test_search(self):\n  c=Config(duration=30,demand=2,disruption_start=10,disruption_end=20)\n  a=optimise(c,4,2);b=optimise(c,4,2)\n  self.assertEqual(a['policy'],b['policy']);Policy(**a['policy']).validate()\n  with self.assertRaises(ValueError):experiment(c,seeds=[42])\nif __name__=='__main__':unittest.main()\n"}
for name, source in FILES.items():
    (RUNTIME / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(RUNTIME))
print("Application files:", RUNTIME)

Application files: /workspace/scratch/14205fdf6793/transit_lab/metro_runtime


## 2. Model and units
Time step: 0.25 minutes. Adjacent stations: 1.2 km; base traversal: 2 minutes. Each directed segment admits one train until traversal plus 0.5 minutes clearance completes. Minimum departure interval is 1.5 minutes. This is a conservative simulation occupancy rule, not a braking or signalling model.

Passenger destinations are sampled downstream. Queues are FIFO. Train capacity is enforced. Trains reverse at terminals after two minutes. No separate reserve pool or train-withdrawal event is implemented. Fleet changes represent scenario-level fleet availability. Platform capacity and interactions between waiting trains are not modelled.

Score = observed queue passenger-minutes / arrivals + 0.2 × energy proxy / arrivals. Boarded-only wait excludes unfinished waiting; report endpoint queues alongside it. The queue integral includes waiting during the observation window; it is not eventual full journey delay.

In [2]:
from transit import Config, Policy, simulate, optimise, experiment
from dataclasses import asdict
config = Config(duration=60, disruption_start=20, disruption_end=40)
result = simulate(config)
print(json.dumps(result["metrics"], indent=2))

{
  "generated": 14135,
  "boarded": 13841,
  "completed": 12295,
  "waiting_at_end": 294,
  "onboard_at_end": 1546,
  "mean_boarded_wait": 2.612708619319413,
  "p95_boarded_wait": 10.0,
  "queue_passenger_minutes": 36646.5,
  "observed_wait_per_arrival": 2.5926070038910507,
  "energy_kwh_proxy": 1870.0860000000002,
  "passenger_km": 46417.20000000002,
  "energy_per_passenger_km": 0.04028864300302473,
  "blocked_departure_attempts": 270,
  "objective": 2.6190673646975595
}


## 3. Validate simulation invariants
Tests cover demand conservation (inside every simulation), deterministic replay, zero demand, capacity, segment clearance and rejected inputs.

In [3]:
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_transit"], cwd=RUNTIME, capture_output=True, text=True)
print(check.stdout + check.stderr)
assert check.returncode == 0

test_reproducible (test_transit.TransitTests.test_reproducible) ... ok
test_search (test_transit.TransitTests.test_search) ... ok
test_separation_and_capacity (test_transit.TransitTests.test_separation_and_capacity) ... ok
test_validation (test_transit.TransitTests.test_validation) ... ok
test_zero_demand (test_transit.TransitTests.test_zero_demand) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.084s

OK



## 4. Evolve bounded scheduling parameters
A seeded elitist GA uses uniform crossover, Gaussian mutation, clipping repair and coordinate refinement. It tunes numerical parameters; it does not discover unrestricted Python programs. Training seeds are reused across candidates to control noise. Search costs are recorded.

In [4]:
search = optimise(config, population=6, generations=3)
print(json.dumps(search, indent=2))
optimised = simulate(config, Policy(**search["policy"]))

{
  "policy": {
    "headway": 3,
    "crowd_weight": 0,
    "hold": 0,
    "eco": 0
  },
  "training_objective": 2.626751002427189,
  "curve": [
    {
      "generation": 0,
      "best": 2.626751002427189
    },
    {
      "generation": 1,
      "best": 2.626751002427189
    },
    {
      "generation": 2,
      "best": 2.626751002427189
    }
  ],
  "evaluations": 32,
  "seconds": 0.6148706300000413,
  "training_seeds": [
    42,
    43
  ]
}


## 5. Optional LLM-assisted rule proposal
Give an external model this prompt: “Propose only JSON with headway [2,6], crowd_weight [0,1], hold [0,2], eco [0,1] for a disrupted single-line metro simulator. Do not claim performance gains. headway is terminal dispatch interval; crowd_weight reduces it under terminal crowding; hold extends dwell in proportion to alighting; eco increases travel time and reduces the energy proxy.”

Paste the response below. Validation rejects extra keys and out-of-range values. The example is manually supplied, not an LLM experimental result. Compare imported-only and imported-seeded evolution against the same budgets before claiming an LLM contribution.

In [5]:
proposal_json = '{"headway":3.2,"crowd_weight":0.7,"hold":0,"eco":0.3}'
proposal = Policy(**json.loads(proposal_json)).validate()
print(asdict(proposal))

{'headway': 3.2, 'crowd_weight': 0.7, 'hold': 0, 'eco': 0.3}


## 6. Independent evaluation
The following demonstration uses three held-out demand seeds. For publication, pre-register additional workload shifts, independent training replicates, matched evaluation budgets and paired confidence intervals. Three seeds alone do not establish statistical significance. Compare plain GA, GA + local search, imported-only, and imported-seeded GA for a full component ablation; only the combined GA + local search is implemented here.

In [6]:
comparison = experiment(config, seeds=(101,102,103), population=4, generations=2, initial=proposal)
from collections import defaultdict
import statistics
groups=defaultdict(list)
for row in comparison["rows"]: groups[row["method"]].append(row["observed_wait_per_arrival"])
for name, values in groups.items(): print(name, "mean observed wait:", round(statistics.mean(values),3), "minutes")

Fixed timetable mean observed wait: 2.576 minutes
Demand reactive mean observed wait: 2.576 minutes
Evolution + local search mean observed wait: 2.576 minutes
Imported LLM rule mean observed wait: 3.039 minutes


## 7. Save reproducible outputs
All outputs include configuration, policy and seeds. Energy units are nominal kWh proxy values until calibrated. Keep simulator version and source with every experiment.

In [7]:
(RUNTIME / "simulation.json").write_text(json.dumps(result, indent=2))
(RUNTIME / "comparison.json").write_text(json.dumps(comparison, indent=2))
print("Saved simulation.json and comparison.json")

Saved simulation.json and comparison.json


## 8. Launch the interactive front end
Run the next cell, then open http://127.0.0.1:8765 on the same computer. It supports parameter editing, simulation, evolutionary search, independent comparison, playback and JSON export. The server deliberately binds to loopback, checks origins, caps request size and serves only explicit paths. Compute requests are serial and bounded. On hosted notebooks, download and run the ZIP locally; localhost refers to the machine running Python.

In [8]:
import subprocess, atexit
server_process = subprocess.Popen([sys.executable, str(RUNTIME / "server.py")], stdout=(RUNTIME / "server.log").open("w"), stderr=subprocess.STDOUT)
atexit.register(server_process.terminate)
print("Dashboard: http://127.0.0.1:8765  PID:", server_process.pid)

Dashboard: http://127.0.0.1:8765  PID: 7


## 9. Stop the server

In [9]:
server_process.terminate()
server_process.wait(timeout=5)
print("Server stopped")

Server stopped


## 10. Path towards operational deployment
1. Replace synthetic demand with anonymised station counts and OD estimates; use temporal train/validation/test separation.
2. Calibrate travel, dwell, capacity and energy against operator records.
3. Extend infrastructure modelling to platforms, junctions, turnbacks, transfers, failures and reserve train insertion.
4. Introduce receding-horizon state carryover and an independent CP-SAT feasibility model; neither is implemented here.
5. Validate under demand shifts and infrastructure disturbances, then operate in shadow mode with operator review.
6. Integrate only through approved supervisory interfaces. Existing signalling and train protection retain movement authority.

Reference direction: AlphaEvolve (https://arxiv.org/abs/2506.13131). This prototype is inspired by evaluator-driven search, not a reproduction of AlphaEvolve. The supplied CSCU9YE notes provide the teaching basis for heuristics, evolutionary search, constraints and fair evaluation.